# Explorative Analyse: Arbeitsmarkt Regensburg (Jobcenter) — Teil 12: Unterbeschäftigung

Offener Punkt aus [`ArbeitslosenAnalyse.md`](../MD/ArbeitslosenAnalyse.md): Das Blatt
"1.1 Eckwerte" enthält neben der bereits genutzten **Unterbeschäftigungsquote** auch die drei
zugrundeliegenden **absoluten Kennzahlen** — bislang nur für die Quote verwendet, nie als
eigene Zeitreihe betrachtet:

- **Arbeitslosigkeit im weiteren Sinne**
- **Unterbeschäftigung im engeren Sinne**
- **Unterbeschäftigung (ohne Kurzarbeit)**

Anders als bei den "gemeldeten Arbeitsstellen" ([`11_Arbeitsstellen.ipynb`](11_Arbeitsstellen.ipynb))
liegt hier für alle drei Kennzahlen **sowohl ein "Insgesamt"- als auch ein SGB-II-Wert** vor —
die Daten sind bereits vollständig und eindeutig in
[`eckwerte_long.csv`](../Data/processed/eckwerte_long.csv) enthalten, ein Neu-Einlesen der
Rohdateien ist hier nicht nötig.

In [1]:
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go

PROCESSED_DIR = Path("..") / "Data" / "processed"
ABBILDUNGEN_DIR = Path("..") / "Docs" / "Abbildungen"

kennzahlen = pd.read_csv(PROCESSED_DIR / "kennzahlen_sgb2.csv")
kennzahlen["period"] = pd.PeriodIndex(kennzahlen["period"], freq="M")
kennzahlen = kennzahlen.set_index("period").sort_index()

df_long = pd.read_csv(PROCESSED_DIR / "eckwerte_long.csv")
df_long["period"] = pd.PeriodIndex(df_long["period"], freq="M")

unterbeschaeftigung = df_long[df_long["kategorie"] == "Unterbeschäftigung"].copy()
unterbeschaeftigung[["period", "merkmal", "insgesamt", "sgb2"]].head(8)

,period,merkmal,insgesamt,sgb2
21,2025-01,Arbeitslosigkeit im weiteren Sinne,3973.0,1665.0
22,2025-01,Unterbeschäftigung (ohne Kurzarbeit),4706.0,2190.0
23,2025-01,Unterbeschäftigung im engeren Sinne,4664.0,2188.0
24,2025-01,Unterbeschäftigungsquote,4.0,1.9
52,2025-02,Arbeitslosigkeit im weiteren Sinne,3912.0,1573.0
53,2025-02,Unterbeschäftigung (ohne Kurzarbeit),4700.0,2134.0
54,2025-02,Unterbeschäftigung im engeren Sinne,4661.0,2132.0
55,2025-02,Unterbeschäftigungsquote,4.0,1.8


## 1. Qualitätssicherung: `drop_duplicates()`

Bevor wir pivotieren, prüfen wir explizit auf doppelte (Berichtsmonat, Merkmal)-Kombinationen
— sollte es beim Einlesen der 21 Dateien zu einem Duplikat gekommen sein, würde `pivot_table()`
das stillschweigend per Aggregation "reparieren" und den Fehler verschleiern.

In [2]:
anzahl_duplikate = unterbeschaeftigung.duplicated(subset=["period", "merkmal"]).sum()
print(f"Gefundene Duplikate (Berichtsmonat, Merkmal): {anzahl_duplikate}")

unterbeschaeftigung = unterbeschaeftigung.drop_duplicates(subset=["period", "merkmal"])
anzahl_monate = unterbeschaeftigung["period"].nunique()
anzahl_merkmale = unterbeschaeftigung["merkmal"].nunique()
print(f"Zeilen nach drop_duplicates(): {len(unterbeschaeftigung)} "
      f"(erwartet: {anzahl_monate} Monate × {anzahl_merkmale} Merkmale = {anzahl_monate * anzahl_merkmale})")

Gefundene Duplikate (Berichtsmonat, Merkmal): 0
Zeilen nach drop_duplicates(): 84 (erwartet: 21 Monate × 4 Merkmale = 84)


## 2. Aufbereitung: Pivotieren, Datentypen, alternative Datumsverarbeitung

Die Longtable wird auf ein Wide-Format mit separaten Spalten für "Insgesamt" und "SGB II" je
Merkmal umgestellt. Die drei Personenzahlen werden anschließend mit `astype()` auf einen
nullable Integer-Typ (`Int64`) umgestellt — sie sind immer ganzzahlig, im Gegensatz zur
Quote. Zusätzlich zeigen wir mit `pd.to_datetime()` eine alternative Datumsverarbeitung, wie
sie in den übrigen Notebooks (die durchgehend `pd.Period` nutzen) bislang nicht vorkam.

In [3]:
wide = unterbeschaeftigung.pivot_table(index="period", columns="merkmal", values=["insgesamt", "sgb2"])
wide = wide.sort_index()

count_spalten = [
    c for c in wide.columns if c[1] != "Unterbeschäftigungsquote"
]
wide[count_spalten] = wide[count_spalten].astype("Int64")

# Alternative zu pd.Period: dieselbe Zeitachse als 'echtes' Datum interpretieren.
wide_ts = wide.copy()
wide_ts.index = pd.to_datetime(wide_ts.index.to_timestamp())
wide.dtypes.head(4)

           merkmal                             
insgesamt  Arbeitslosigkeit im weiteren Sinne        Int64
           Unterbeschäftigung (ohne Kurzarbeit)      Int64
           Unterbeschäftigung im engeren Sinne       Int64
           Unterbeschäftigungsquote                float64
dtype: object

## 3. Zeitreihe der drei Komponenten (SGB II)

Wie entwickeln sich die drei zugrundeliegenden Personenzahlen im Zeitverlauf, jeweils für den
Rechtskreis SGB II?

In [4]:
fig = go.Figure()
farben = {"Arbeitslosigkeit im weiteren Sinne": "#2563eb", "Unterbeschäftigung im engeren Sinne": "#7c3aed", "Unterbeschäftigung (ohne Kurzarbeit)": "#dc2626"}
for merkmal, farbe in farben.items():
    fig.add_trace(go.Scatter(
        x=wide_ts.index, y=wide_ts[("sgb2", merkmal)], name=merkmal,
        mode="lines+markers", line=dict(color=farbe, width=2), marker=dict(size=5),
    ))
fig.update_layout(
    title="Unterbeschäftigung (SGB II): drei Abgrenzungen im Vergleich",
    xaxis_title="Berichtsmonat", yaxis_title="Personen", template="plotly_white",
    height=450, hovermode="x unified", legend=dict(orientation="h", y=1.15),
)
fig.write_image(ABBILDUNGEN_DIR / "abb36_unterbeschaeftigung_komponenten.png", scale=2)
fig

In [5]:
kurzarbeit_komponente = wide[("sgb2", "Unterbeschäftigung (ohne Kurzarbeit)")] - wide[("sgb2", "Unterbeschäftigung im engeren Sinne")]
massnahmen_komponente = wide[("sgb2", "Unterbeschäftigung im engeren Sinne")] - wide[("sgb2", "Arbeitslosigkeit im weiteren Sinne")]

print("Kurzarbeiter-Komponente (SGB II) = 'ohne Kurzarbeit' minus 'im engeren Sinne':")
print(kurzarbeit_komponente.describe()[["mean", "min", "max"]])
print()
print("Maßnahmen-Komponente (SGB II) = 'im engeren Sinne' minus 'Arbeitslosigkeit im weiteren Sinne':")
print(massnahmen_komponente.describe()[["mean", "min", "max"]])

Kurzarbeiter-Komponente (SGB II) = 'ohne Kurzarbeit' minus 'im engeren Sinne':
mean    2.380952
min          1.0
max          4.0
dtype: Float64

Maßnahmen-Komponente (SGB II) = 'im engeren Sinne' minus 'Arbeitslosigkeit im weiteren Sinne':
mean    388.428571
min          314.0
max          559.0
dtype: Float64


**Befund:** Die Kurzarbeiter-Komponente ist im SGB-II-Kontext verschwindend gering — im Mittel nur 2,4 Personen (Spanne 1 bis 4) über alle 21 Monate. Kurzarbeit spielt für Bürgergeld-Beziehende in Regensburg damit praktisch keine Rolle. Die Maßnahmen-Komponente (Teilnehmende an arbeitsmarktpolitischen Maßnahmen, die nicht als arbeitslos gemeldet sind) ist dagegen deutlich substanzieller: im Mittel 388 Personen (Spanne 314–559) — rund ein Fünftel zusätzlich zur reinen Arbeitslosenzahl. In den letzten Monaten liegt sie mit 315–343 Personen allerdings am unteren Rand dieser Spanne.

## 4. SGB-II-Anteil im Vergleich zu Bestand und Langzeitarbeitslosigkeit

Der Rechtskreisvergleich in [`10_Rechtskreisvergleich.ipynb`](10_Rechtskreisvergleich.ipynb)
zeigte einen Jobcenter-Anteil von rund 40 % am gesamten Arbeitslosenbestand und rund 72 % an
der Langzeitarbeitslosigkeit. Wo liegt die Unterbeschäftigung dazwischen?

In [6]:
anteil = (wide[("sgb2", "Unterbeschäftigung im engeren Sinne")] / wide[("insgesamt", "Unterbeschäftigung im engeren Sinne")] * 100)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=wide_ts.index, y=anteil, mode="lines+markers", name="SGB-II-Anteil: Unterbeschäftigung (engere Sinne)",
    line=dict(color="#059669", width=2.5),
))
fig.add_hline(y=39.7, line_dash="dot", line_color="#2563eb", annotation_text="Ø Anteil am Bestand (≈ 39,7 %)")
fig.add_hline(y=72.2, line_dash="dot", line_color="#dc2626", annotation_text="Ø Anteil an LZA (≈ 72,2 %)")
fig.update_layout(
    title="SGB-II-Anteil an der Unterbeschäftigung im engeren Sinne, im Vergleich",
    xaxis_title="Berichtsmonat", yaxis_title="Anteil (%)", yaxis_ticksuffix="%",
    template="plotly_white", height=450,
)
fig.write_image(ABBILDUNGEN_DIR / "abb37_unterbeschaeftigung_anteil.png", scale=2)
fig

**Befund:** Der SGB-II-Anteil an der Unterbeschäftigung im engeren Sinne liegt bei 42,7 % (September 2026, im Januar 2025 noch 46,9 %), mit einer Spanne von 41,1 % bis 49,6 % über den gesamten Zeitraum — und ordnet sich damit genau zwischen den beiden bereits bekannten Werten ein: näher am Anteil des Arbeitslosenbestands (≈ 39,7 %) als am Anteil der Langzeitarbeitslosigkeit (≈ 72,2 %). Der Anteil ist über den Zeitraum leicht rückläufig. Das passt ins Bild: Unterbeschäftigung zählt zusätzlich Maßnahmen-Teilnehmende, ist aber noch nicht so stark auf das SGB II konzentriert wie die Langzeitarbeitslosigkeit.

## 5. Zusammenfassung

Die drei bislang nur für die Unterbeschäftigungsquote genutzten absoluten Kennzahlen zeigen als eigene Zeitreihe zwei neue Erkenntnisse: Erstens ist die Kurzarbeiter-Komponente im SGB II praktisch bedeutungslos (1–4 Personen), während die Maßnahmen-Komponente mit durchschnittlich 388 Personen ein spürbarer Aufschlag auf die reine Arbeitslosenzahl ist. Zweitens liegt der SGB-II-Anteil an der Unterbeschäftigung (41–50 %) zwischen dem bereits bekannten Anteil am Arbeitslosenbestand (≈ 40 %) und dem deutlich höheren Anteil an der Langzeitarbeitslosigkeit (≈ 72 %) — ein weiterer Baustein im Gesamtbild, wie unterschiedlich stark sich verschiedene Kennzahlen auf den Rechtskreis SGB II konzentrieren.

Methodisch schließt dieses Notebook die beiden letzten noch offenen Punkte aus [`ThemenCheck.md`](../MD/ThemenCheck.md): `drop_duplicates()` als expliziter Qualitätscheck vor dem Pivotieren, sowie `astype()` und `pd.to_datetime()` als alternative Datentyp- bzw. Datumsverarbeitung.

In [7]:
export = wide["sgb2"].rename(columns={
    "Arbeitslosigkeit im weiteren Sinne": "arbeitslosigkeit_weiterer_sinn",
    "Unterbeschäftigung (ohne Kurzarbeit)": "unterbeschaeftigung_ohne_kurzarbeit",
    "Unterbeschäftigung im engeren Sinne": "unterbeschaeftigung_engerer_sinn",
    "Unterbeschäftigungsquote": "unterbeschaeftigungsquote",
}).copy()
# Zusaetzlich den "Insgesamt"-Wert fuer eine Kennzahl mitspeichern, damit sich der
# SGB-II-Anteil (siehe Abschnitt 4) auch ausserhalb dieses Notebooks berechnen laesst.
export["unterbeschaeftigung_engerer_sinn_insgesamt"] = wide[("insgesamt", "Unterbeschäftigung im engeren Sinne")]
export.index.name = "period"
export.to_csv(PROCESSED_DIR / "unterbeschaeftigung_sgb2.csv")
print("Gespeichert:", PROCESSED_DIR / "unterbeschaeftigung_sgb2.csv")

Gespeichert: ..\Data\processed\unterbeschaeftigung_sgb2.csv
